# GraphSAGE link prediction on the F1 KGs

Three target queries (h, r, ?), each a domain fact that is identical in all three KGs
(`adhoc` / `pure_sulo` / `hybrid_sulo`), whatever predicates the KG spells it with:

| Target | Query | Fact | Candidates |
|---|---|---|---|
| Team | `RaceEntry → ?` | `hasTeam` | 182 Teams |
| Position | `RaceEntry → ?` | `hasStartPosition` (grid slot) | 34 Positions |
| Circuit | `Race → ?` | `heldAtCircuit` (Hybrid: `isInCircuit`) | 78 Circuits |

One joint model per KG: a GraphSAGE-mean encoder (`GraphSAGEModel`, relation types ignored) and a DistMult decoder with one vector per target, trained
with 1-vs-all cross-entropy over each query's candidates (every node of the answer class; the
targets are functional, so the filtered ranking is the raw one). Model selection uses pooled val
MRR over all queries; MRR and Hits@1/3/10 are reported per target, mean ± std over `FINAL_SEEDS`,
and compared across KGs within a target only.

The shared logic lives in `link_prediction/link_prediction_tools.py`, `node_prediction/models.py`
and `f1_node_data.py`. This notebook and its R-GCN counterpart differ only in the encoder.
`link_prediction_setup.ipynb` holds the data checks and the simple baselines.

In [1]:
# Must run before CUDA is initialised: avoids fragmentation OOMs across a long sweep.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from dataclasses import replace

import pandas as pd
import torch

from f1_node_data import load_population
from link_prediction.link_prediction_tools import (
    TARGETS, LinkPredictor, baseline_hits_at_1, build_lp_data, evaluate_lp_on, frequency_baseline, race_split,
    run_lp_search, target_convergence, train_lp_model,
)
from node_prediction.gnn_tools import (
    cleanup, finished_tuning_results, limit_cuda_memory_to_free_vram, runs_ended_by_epoch_cap, set_seed,
)
from node_prediction.models import GraphSAGEModel

KGS = ["adhoc", "pure_sulo", "hybrid_sulo"]
OUT_DIR = ROOT / "outputs" / "f1" / "link_prediction" / "graphsage"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(torch.__version__, "| CUDA:", torch.cuda.is_available())
# WSL2 would otherwise overcommit VRAM into RAM and fail with a driver error instead of an OOM,
# which the sweep could not skip.
if torch.cuda.is_available():
    # Close other GPU programs first: the largest config peaks at 7.8 GiB (R-GCN Pure SULO 256-d x 3,
    # 20 bases; GraphSAGE 256-d x 3 less), so a lower cap OOMs it, which the tuning cell then reports.
    print(f"CUDA memory cap: {limit_cuda_memory_to_free_vram() / 2**30:.2f} GiB")

/home/domin/miniconda3/envs/sulo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.7.0+cu126 | CUDA: True
CUDA memory cap: 10.48 GiB


## Data

`build_lp_data` renders each KG from `data/f1/assertions.csv` **without** `hasFinishPosition`, for
every `RaceEntry`: in Pure SULO the finish and start position are the same
`sulo:hasFeature → Position/<n>` triple whenever they are equal (1,976 entries), so a held-out grid
slot would otherwise stay in the graph. Every other outcome (points, status, laps) stays.

Holdout is per race (80/10/10 of the 1,153 races, `random_state=42`, shared by all KGs and both
encoders): a val/test race loses the facts of all three targets from message passing, along their
whole path (Pure's team: the `RaceEntry/<id>/hasTeam` Role node and its edges). A `hasTeamMember`
pair (Hybrid/Pure) is shown only while one of the entries behind it shows its team. During
training, each epoch hides and supervises a random `DISJOINT_TRAIN_RATIO` share of the training
races, so training queries look like test queries. Node features are those of node prediction;
the degree feature is recomputed on each visible graph.

In [2]:
entities, assertions = load_population(ROOT / "data" / "f1")
split = race_split(entities, seed=42)
lp_data = {kg: build_lp_data(kg, entities, assertions, split) for kg in KGS}

def facts_per_split(t, d):
    counts = torch.bincount(d.race_split[t.race], minlength=3).tolist()
    return " / ".join(map(str, counts))

pd.DataFrame({
    kg: {
        "nodes": d.data.num_nodes,
        "edges (fwd + inv)": d.data.edge_index.size(1),
        "relation types (fwd + inv)": d.num_relations,
        "feature keys": len(d.vocabulary),
        "races train / val / test": " / ".join(str(len(p)) for p in split),
        **{f"{name} facts train / val / test": facts_per_split(t, d) for name, t in d.targets.items()},
        "hasTeamMember pairs": d.member_edges.size(1) // 2,
    }
    for kg, d in lp_data.items()
}).T

,nodes,edges (fwd + inv),relation types (fwd + inv),feature keys,races train / val / test,team facts train / val / test,position facts train / val / test,circuit facts train / val / test,hasTeamMember pairs
adhoc,58154,594834,58,1392,922 / 115 / 116,21658 / 2639 / 2735,20321 / 2504 / 2575,922 / 115 / 116,0
pure_sulo,230214,1375418,24,1408,922 / 115 / 116,21658 / 2639 / 2735,20321 / 2504 / 2575,922 / 115 / 116,1941
hybrid_sulo,58193,687354,64,1398,922 / 115 / 116,21658 / 2639 / 2735,20321 / 2504 / 2575,922 / 115 / 116,1941


## Tuning constants

The search and its constants are those of node prediction (`node_prediction/graph_sage.ipynb`), with two
link-prediction changes: the dim grid stops at 256, and `DISJOINT_TRAIN_RATIO` sets the share of
training races supervised per epoch. The same values are used for every KG. GraphSAGE ignores relation types, so there is no bases phase (`grid_bases=[None]`).

In [3]:
# Hidden dims = embedding dims: Hamilton et al. 2017 use 256-d outputs at every depth.
#   https://arxiv.org/abs/1706.02216
# Capped at 256 (user, 2026-11-02), unlike node prediction's 512: the link-prediction graph keeps the
# outcome nodes (Pure SULO 230k nodes), and RGCNConv keeps an N x d activation per relation, so R-GCN
# 512-d does not fit on Pure (24 x 230k x 512 x 4 B ~ 11 GB > the 10.5 GiB cap); 256-d x 3 peaks at
# 7.8 GiB. Both encoders and every KG get the same grid.
GRID_1A_DIM = [32, 64, 128, 256]

# Layers (full-batch, so no neighbour fanouts): 1 vs 2 = Hamilton et al. 2017, Section 4.3; 2 =
# Schlichtkrull et al. 2018; 3 = the depth of PyG's GraphSAGE ogbn-products example. DistMult scores
# <e_head, w_r, e_candidate>, so head and candidate each bring K hops: the driver's other entries
# (RaceEntry -> Driver -> RaceEntry' -> Team) are 3 hops in Ad-hoc/Hybrid, 6 in Pure (Role nodes),
# and Hybrid/Pure add Driver -> Team through hasTeamMember.
#   https://arxiv.org/abs/1703.06103
#   https://arxiv.org/abs/1706.02216
#   https://github.com/pyg-team/pytorch_geometric/blob/2.3.0/examples/ogbn_products_sage.py
GRID_1B_DEPTH = [1, 2, 3]

# Learning rate: Hamilton et al. 2017 searched {1e-2, 1e-3, 1e-4} (supervised); Schlichtkrull et al. 2018
# and PyG's R-GCN example use 1e-2.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/examples/rgcn_link_pred.py
GRID_2B_LR = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

# Weight decay (Adam): Schlichtkrull et al. 2018 searched L2 in {0, 5e-4}; GraphGym (You et al. 2020)
# defaults to 5e-4.
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/torch_geometric/graphgym/config.py
GRID_2C_WD = [0, 1e-5, 1e-4, 5e-4, 1e-3]

# Defaults while earlier phases run (phase 1 tunes depth jointly with dim, so DEFAULT_DEPTH is only
# used by the shuffled-answer check): 2 layers = Schlichtkrull et al. 2018 / Hamilton et al. 2017 (K=2);
# lr 1e-2 = Kipf & Welling 2017 and Schlichtkrull et al. 2018, both full-batch with Adam.
# wd 1e-4 has no direct literature value (between R-GCN's 0 and 5e-4).
#   https://arxiv.org/abs/1609.02907
#   https://arxiv.org/abs/1703.06103
DEFAULT_DEPTH = 2
DEFAULT_LR = 1e-2
DEFAULT_WD = 1e-4

# Full-batch, so one epoch = one gradient step. Max epochs 400 = the top of GraphGym's design space
# {100, 200, 400} (You et al. 2020, Section 4). Patience 50: no literature value (user choice,
# 2026-10-21, as in node prediction). The final retrain uses best_epoch + 1 epochs regardless.
#   https://arxiv.org/abs/2011.08843
EPOCHS = 400
PATIENCE = 50

# Early-stopping warm-up (user, 2026-10-29, as in node prediction): epochs 0-49 are trained but neither
# checkpointed nor counted towards patience, as Keras' EarlyStopping(start_from_epoch). No literature value.
#   https://keras.io/api/callbacks/early_stopping/
START_FROM_EPOCH = 50

# Node input embedding size: the GraphSAGE reference implementation recommends learned identity
# features of 64-256 dims (README, --identity_dim). Also the embedding dim of a 1-layer model.
#   https://github.com/williamleif/GraphSAGE#readme
FEATURE_DIM = 256

# Dropout: PyG's GraphSAGE ogbn-products example uses p=0.5; Hamilton et al. 2017 do not
# report dropout.
#   https://github.com/pyg-team/pytorch_geometric/blob/2.3.0/examples/ogbn_products_sage.py
DROPOUT = 0.5

# Share of the training races whose target facts are hidden from message passing and supervised in an
# epoch (redrawn every epoch; the rest are context). User choice (2026-11-02): 0.3 is the value of PyG's
# link-prediction tutorial for RandomLinkSplit(disjoint_train_ratio), which exists to keep supervision
# edges out of message passing; R-GCN's link-prediction edge dropout of 0.4 is the closest value from
# Schlichtkrull et al. 2018.
#   https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.transforms.RandomLinkSplit.html
#   https://arxiv.org/abs/1703.06103
DISJOINT_TRAIN_RATIO = 0.3

# Decoder and loss (user, 2026-11-02): DistMult (Yang et al. 2015), the R-GCN link-prediction decoder, with
# 1-vs-all cross-entropy over each query's candidates (Dettmers et al. 2018; Ruffinelli et al. 2020 found
# 1vsAll/KvsAll + CE the best training type for most models), every query weighted equally (no
# per-target weight, so Circuit's 1,153 facts weigh ~2%). No negative sampling, hence no ratio constant.
#   https://arxiv.org/abs/1412.6575
#   https://arxiv.org/abs/1707.01476
#   https://openreview.net/forum?id=BkxSmlBFvr

# Every tuning config is trained once per seed and scored by its mean val MRR; the final retrain uses
# the median per-seed best epoch. 3 seeds: user choice (2026-10-24), no literature value.
TUNING_SEEDS = [0, 1, 2]
FINAL_SEEDS = [0, 1, 2, 3, 4]   # final retrain + test, reported as mean ± std

missing = [name for name, value in dict(globals()).items() if name.isupper() and value is None]
assert not missing, f"Set these before training: {missing}"

In [4]:
def model_factory(d):
    """(hidden_list, bases) -> fresh LinkPredictor for one KG; everything KG-specific is bound here.
    The embedding dim is the last hidden dim; a 1-layer model has none and embeds in FEATURE_DIM.
    GraphSAGE has no relation bases: `bases` is always None and ignored."""
    def build(hidden_list, bases):
        dim = hidden_list[-1] if hidden_list else FEATURE_DIM
        encoder = GraphSAGEModel(
            num_keys=len(d.vocabulary), feature_dim=FEATURE_DIM, hidden_channels_list=hidden_list,
            out_channels=dim, dropout=DROPOUT,
        )
        return LinkPredictor(encoder, len(TARGETS), dim)
    return build

def bases_grid(d):
    return [None]

def default_bases(d):
    return None

## Pipeline check: shuffled answers

Trains the default configuration (smallest grid dim) on each real graph with every target's answers
permuted across its facts. The graph then says nothing about a query's answer, so the test MRR must
stay at the frequency baseline (ranking candidates by how often they answered earlier races):
otherwise the answer tensor itself reaches the model's inputs. It cannot detect a held-out *true*
fact left in the graph (that would not predict the shuffled answers); that is checked by
`leaked_facts` in `link_prediction_setup.ipynb` and in `tests/test_link_prediction.py`.

In [5]:
control = {}
for kg, d in lp_data.items():
    generator = torch.Generator().manual_seed(0)
    shuffled = replace(d, targets={n: replace(t, answer=t.answer[torch.randperm(len(t.answer), generator=generator)])
                                   for n, t in d.targets.items()})
    set_seed(TUNING_SEEDS[0])
    model = model_factory(d)([GRID_1A_DIM[0]] * (DEFAULT_DEPTH - 1), default_bases(d))
    model, _ = train_lp_model(model, shuffled, epochs=EPOCHS, lr=DEFAULT_LR, weight_decay=DEFAULT_WD,
                              patience=PATIENCE, disjoint_ratio=DISJOINT_TRAIN_RATIO, seed=TUNING_SEEDS[0],
                              start_from_epoch=START_FROM_EPOCH)
    test, chance = evaluate_lp_on(model, shuffled, "test"), frequency_baseline(shuffled, "test")
    control[kg] = {f"{t} {kind}": m[t]["mrr"] for kind, m in (("gnn", test), ("frequency", chance)) for t in m}
    del model, shuffled
    cleanup()

pd.DataFrame(control).T.round(4)

Epoch 000 | Train Loss: 4.4367 | Val MRR: 0.0930 (team 0.0309, position 0.1576, circuit 0.1112) | Grad Norm: 2.1169
Epoch 001 | Train Loss: 4.9912 | Val MRR: 0.1797 (team 0.2062, position 0.1572, circuit 0.0636) | Grad Norm: 8.9456
Epoch 002 | Train Loss: 4.1489 | Val MRR: 0.1828 (team 0.2060, position 0.1598, circuit 0.1533) | Grad Norm: 2.1981
Epoch 003 | Train Loss: 4.0948 | Val MRR: 0.1805 (team 0.2011, position 0.1597, circuit 0.1600) | Grad Norm: 1.6421
Epoch 004 | Train Loss: 3.9784 | Val MRR: 0.1811 (team 0.2020, position 0.1597, circuit 0.1675) | Grad Norm: 1.6057
Epoch 005 | Train Loss: 4.0487 | Val MRR: 0.1772 (team 0.1943, position 0.1595, circuit 0.1677) | Grad Norm: 3.4448
Epoch 006 | Train Loss: 3.8582 | Val MRR: 0.1547 (team 0.1496, position 0.1594, circuit 0.1688) | Grad Norm: 0.8042
Epoch 007 | Train Loss: 3.9047 | Val MRR: 0.1546 (team 0.1496, position 0.1593, circuit 0.1674) | Grad Norm: 1.5546
Epoch 008 | Train Loss: 3.8775 | Val MRR: 0.1554 (team 0.1495, position 

,team gnn,position gnn,circuit gnn,pooled gnn,team frequency,position frequency,circuit frequency,pooled frequency
adhoc,0.2031,0.1691,0.1811,0.1865,0.2085,0.1625,0.1784,0.186
pure_sulo,0.2089,0.1534,0.1809,0.1819,0.2085,0.1625,0.1784,0.186
hybrid_sulo,0.2014,0.1657,0.1763,0.1839,0.2085,0.1625,0.1784,0.186


## Sequential tuning and seeded final evaluation

Per KG, node prediction's search on pooled val MRR: a joint dim × depth grid, then greedy bases → lr →
wd at the winning architecture, then a 2×2 verification grid over the top-2 dims (at the winning
depth) × the top-2 regularisation combos. Every configuration is trained once per seed in
`TUNING_SEEDS` and scored by its mean val MRR (per-seed values in the CSV). The winner is retrained
on the train + val races for its median best epoch + 1 epochs and tested once per seed in
`FINAL_SEEDS`; the test races are then scored with the train + val facts as context.

In [6]:
tuning = {}
for kg, d in lp_data.items():
    csv_path = OUT_DIR / f"{kg}_sequential_tuning_results.csv"
    # Resume after a crash: a KG whose search already finished with these grids and seeds is loaded, not re-run.
    tuning[kg] = finished_tuning_results(csv_path, GRID_1A_DIM, GRID_1B_DEPTH, TUNING_SEEDS, FINAL_SEEDS,
                                         metric="val_mrr")
    if tuning[kg] is not None:
        print(f"{kg}: finished results loaded from {csv_path}")
        continue
    tuning[kg] = run_lp_search(
        model_factory=model_factory(d), lp=d, kg=kg, save_dir=f"{OUT_DIR / kg}/", csv_path=csv_path,
        grid_dim=GRID_1A_DIM, grid_depth=GRID_1B_DEPTH, grid_bases=bases_grid(d), grid_lr=GRID_2B_LR,
        grid_wd=GRID_2C_WD, default_bases=default_bases(d), default_lr=DEFAULT_LR, default_wd=DEFAULT_WD,
        epochs=EPOCHS, patience=PATIENCE, start_from_epoch=START_FROM_EPOCH,
        disjoint_ratio=DISJOINT_TRAIN_RATIO, tuning_seeds=TUNING_SEEDS, final_seeds=FINAL_SEEDS,
    )
    cleanup()

# OOM-ed configs are skipped, not fatal: if any KG lost one, the KGs were not tuned over the same grid.
failed = pd.concat(df[df["status"] != "ok"].assign(kg=kg) for kg, df in tuning.items())
assert failed.empty, f"Tuning runs that did not finish:\n{failed[['kg', 'phase', 'parameter_tested', 'status']]}"
trained = pd.concat(df[df["runtime_sec"] > 0].dropna(subset=["best_epoch_seeds"]).assign(kg=kg) for kg, df in tuning.items())
early = trained[trained["best_epoch_seeds"].map(lambda e: min(int(x) for x in str(e).split(";")) < START_FROM_EPOCH)]
assert early.empty, f"Tuning runs selected inside the warm-up:\n{early[['kg', 'phase', 'best_epoch_seeds']]}"


=== Link-prediction search: adhoc (seeds [0, 1, 2]) ===
Epoch 000 | Train Loss: 4.2706 | Val MRR: 0.1859 (team 0.2132, position 0.1587, circuit 0.1482) | Grad Norm: 3.6345
Epoch 001 | Train Loss: 4.4476 | Val MRR: 0.1390 (team 0.2131, position 0.0606, circuit 0.1461) | Grad Norm: 3.3257
Epoch 002 | Train Loss: 19.1346 | Val MRR: 0.1858 (team 0.2132, position 0.1587, circuit 0.1469) | Grad Norm: 39.6738
Epoch 003 | Train Loss: 5.0679 | Val MRR: 0.1854 (team 0.2139, position 0.1571, circuit 0.1482) | Grad Norm: 4.5713
Epoch 004 | Train Loss: 6.6252 | Val MRR: 0.1850 (team 0.2131, position 0.1571, circuit 0.1487) | Grad Norm: 5.4250
Epoch 005 | Train Loss: 6.9630 | Val MRR: 0.1286 (team 0.1008, position 0.1571, circuit 0.1478) | Grad Norm: 5.2623
Epoch 006 | Train Loss: 6.8784 | Val MRR: 0.0984 (team 0.0407, position 0.1571, circuit 0.1425) | Grad Norm: 5.1112
Epoch 007 | Train Loss: 6.1877 | Val MRR: 0.0899 (team 0.0236, position 0.1571, circuit 0.1478) | Grad Norm: 5.3928
Epoch 008 | T

## Results

Test metrics of the final model per KG and target, mean ± std over `FINAL_SEEDS`, next to two
baselines that see the train + val races only: ranking by answer frequency (MRR), and the one-rule
predictors of `baseline_hits_at_1` (Hits@1: the driver's team that season, the race name's circuit,
the qualifying position).

In [7]:
final = pd.concat(df[df["phase"] == "4_Final_Test_Evaluation"].assign(kg=kg) for kg, df in tuning.items())
metrics = [f"test_{t}_{m}" for t in TARGETS for m in ("mrr", "hits@1", "hits@3", "hits@10")] + ["test_mrr"]
summary = final.groupby("kg")[metrics].agg(["mean", "std"]).round(4).reindex(KGS)
summary.to_csv(OUT_DIR / "final_test_summary.csv")

frequency = frequency_baseline(lp_data["adhoc"], "test")   # the same facts in every KG
rules = baseline_hits_at_1(assertions, split)
display(pd.DataFrame({t: {"frequency MRR": frequency[t]["mrr"], "frequency Hits@1": frequency[t]["hits@1"],
                          "one-rule Hits@1": rules[t]} for t in TARGETS}).round(4))
summary

,team,position,circuit
frequency MRR,0.2072,0.1655,0.1642
frequency Hits@1,0.0892,0.0450,0.0517
one-rule Hits@1,0.9707,0.4299,0.8621


test_team_mrr         test_team_hits@1         test_team_hits@3  \
                     mean     std             mean     std             mean   
kg                                                                            
adhoc              0.3973  0.0152           0.2301  0.0136           0.4575   
pure_sulo          0.2124  0.0120           0.0790  0.0110           0.2168   
hybrid_sulo        0.3291  0.0073           0.1569  0.0075           0.3857   

                    test_team_hits@10         test_position_mrr          ...  \
                std              mean     std              mean     std  ...   
kg                                                                       ...   
adhoc        0.0227            0.7650  0.0221            0.2576  0.0028  ...   
pure_sulo    0.0189            0.5463  0.0092            0.2163  0.0038  ...   
hybrid_sulo  0.0085            0.7269  0.0150            0.2508  0.0048  ...   

            test_circuit_mrr         test_circuit_hits@1          \
                        mean     std                mean     std   
kg                                                                 
adhoc                 0.5550  0.0587              0.3724  0.0649   
pure_sulo             0.2860  0.0521              0.1172  0.0277   
hybrid_sulo           0.3376  0.0565              0.1534  0.0611   

            test_circuit_hits@3         test_circuit_hits@10         test_mrr  \
                           mean     std                 mean     std     mean   
kg                                                                              
adhoc                    0.6862  0.0984               0.9103  0.0359   0.3344   
pure_sulo                0.3397  0.0855               0.6586  0.1036   0.2158   
hybrid_sulo              0.3810  0.0614               0.7983  0.0540   0.2921   

                     
                std  
kg                   
adhoc        0.0092  
pure_sulo    0.0056  
hybrid_sulo  0.0060  

[3 rows x 26 columns]

## Per-target convergence of the winners

Selection and early stopping use pooled val MRR, where Circuit's 115 val races weigh ~2%. For each
KG's winner and tuning seed this lists every target's val MRR at the selected epoch and its own
peak: a target that peaks well after the selected epoch was still improving when the pooled score
chose (read from the per-epoch history in the tuning checkpoints).

In [8]:
config = ["hidden_dim", "layers", "num_bases", "lr", "weight_decay"]
convergence = []
for kg, df in tuning.items():
    win = df.loc[df["phase"] == "4_Final_Test_Evaluation", config].iloc[0]
    phase_3 = df[df["phase"].str.startswith("3_") & (df["status"] == "ok")]
    row = phase_3[(phase_3[config].fillna(-1) == win.fillna(-1)).all(axis=1)].iloc[0]
    convergence.append(target_convergence(f"{OUT_DIR / kg}/", kg, row, TUNING_SEEDS, START_FROM_EPOCH))
convergence = pd.concat(convergence)
convergence.assign(gap=convergence["peak_val_mrr"] - convergence["val_mrr_at_selected"]).round(4)

/tmp/ipykernel_1857298/1439853004.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row = phase_3[(phase_3[config].fillna(-1) == win.fillna(-1)).all(axis=1)].iloc[0]
/tmp/ipykernel_1857298/1439853004.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row = phase_3[(phase_3[config].fillna(-1) == win.fillna(-1)).all(axis=1)].iloc[0]
/tmp/ipykernel_1857298/1439853004.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to

,kg,seed,target,selected_epoch,val_mrr_at_selected,peak_epoch,peak_val_mrr,gap
0,adhoc,0,team,397,0.3949,391,0.3992,0.0043
1,adhoc,0,position,397,0.2694,331,0.2729,0.0036
2,adhoc,0,circuit,397,0.5383,386,0.5696,0.0313
3,adhoc,1,team,388,0.3904,388,0.3904,0.0000
4,adhoc,1,position,388,0.2658,381,0.2690,0.0032
5,adhoc,1,circuit,388,0.5061,394,0.5096,0.0036
6,adhoc,2,team,396,0.3999,399,0.4004,0.0005
7,adhoc,2,position,396,0.2677,380,0.2737,0.0061
8,adhoc,2,circuit,396,0.5029,350,0.5630,0.0601
0,pure_sulo,0,team,168,0.2315,192,0.2336,0.0021


## Limitation: runs ended by the epoch cap

A run stops early at `best_epoch + PATIENCE`, so a seed whose best epoch is ≥ `EPOCHS − PATIENCE`
was cut off by the `EPOCHS` cap while possibly still improving. `EPOCHS = 400` is kept at the top of
GraphGym's design space, as in node prediction, so this is reported as a limitation. The table lists
every trained tuning configuration with at least one capped seed.

In [9]:
capped = pd.concat(runs_ended_by_epoch_cap(df, EPOCHS, PATIENCE).assign(kg=kg) for kg, df in tuning.items())
winners = final.drop_duplicates("kg")[["kg", "hidden_dim", "layers", "num_bases", "lr", "weight_decay", "best_epoch"]]
display(winners.assign(winner_capped=winners["best_epoch"] >= EPOCHS - PATIENCE))
capped[["kg", "phase", "parameter_tested", "val_mrr", "val_mrr_seeds", "best_epoch_seeds"]]

,kg,hidden_dim,layers,num_bases,lr,weight_decay,best_epoch,winner_capped
24,adhoc,256.0,3,None,0.005,0.0,396,True
24,pure_sulo,64.0,3,None,0.010,0.0,210,False
24,hybrid_sulo,64.0,3,None,0.005,0.0,392,True


,kg,phase,parameter_tested,val_mrr,val_mrr_seeds,best_epoch_seeds
1,adhoc,1_Architecture,layers_2_dim_32,0.216693,0.2259;0.2121;0.2121,368;95;145
3,adhoc,1_Architecture,layers_2_dim_128,0.227086,0.2278;0.2207;0.2328,392;118;335
5,adhoc,1_Architecture,layers_3_dim_32,0.222850,0.2141;0.2004;0.2541,90;56;397
6,adhoc,1_Architecture,layers_3_dim_64,0.254465,0.2532;0.2641;0.2462,329;391;388
7,adhoc,1_Architecture,layers_3_dim_128,0.230150,0.1934;0.2274;0.2696,54;190;382
8,adhoc,1_Architecture,layers_3_dim_256,0.230405,0.2165;0.2195;0.2552,148;197;396
12,adhoc,2b_LR,lr_0.001,0.235800,0.2335;0.2351;0.2388,392;375;399
13,adhoc,2b_LR,lr_0.005,0.266259,0.2625;0.2615;0.2748,383;360;357
15,adhoc,2c_WD,wd_0,0.304497,0.3076;0.2923;0.3135,399;397;389
16,adhoc,2c_WD,wd_1e-05,0.278564,0.2693;0.2827;0.2837,375;397;389
